# 39 · Platform patterns, frontier and case studies
Level L6 · Part 6 · deep · ~20 min · builds on 35, 37

Time: about 20 min. This is a **concept** lesson: no server, no Docker, no key. It has two small arithmetic cells you can check by hand. The release dates are the dates on the GitHub release pages (opened 2026-10-21 to 2026-10-23). Vendor and project numbers are labelled as such. Everything else in the lesson comes from the sources named in the text.

## Goal
By the end you can write an SLO and an error budget for a shared LLM platform and name the owner of each decision, date the recent serving changes and say how mature each one is, and summarise three published write-ups with their own numbers.

## Why it matters
- **Without it:** every team runs its own engine, picks its own latency target and sets its own alarms. Nobody can say whether the platform is healthy, so each incident starts with an argument about whose number is right. A new engine release arrives every few weeks, and each team decides alone whether to try it.
- **With it:** one shared set of measures (SLIs), one set of targets (SLOs), one error budget per tenant tier, and one named owner per decision. A release is judged against the same budget as every other change.
- **The cost of skipping it:** the first quarter goes to arguing about the dashboard, and the second to a release that slowed the one tenant with long prompts, because the average hid it.
- **When you don't need it:** one team, one model, one engine. One written latency target is enough.

## The idea
A shared platform is a service with several customers. It runs on three things: a few measures everyone agrees to read (SLIs), targets for those measures with a budget for failure (SLOs and error budgets), and a named owner for each decision. The engine underneath changes fast, so each change is dated and rated for maturity. Think of a city water utility. It agrees a pressure target with each district, allows a fixed number of outages per year, and names the crew that owns each pipe.

## Picture
```mermaid
flowchart LR
  T1["Product team A: prompts, tenant budget, SLO target"] --> G
  T2["Product team B: prompts, tenant budget, SLO target"] --> G
  G["Platform: gateway, routing, autoscaling, golden-set gate"] --> E["Engine pools (vLLM, SGLang)"]
  E --> M["Metrics per tenant tier: TTFT, TPOT, success, goodput"]
  M --> B{"Error budget left?"}
  B -- "yes" --> R["Ship the next change"]
  B -- "no" --> F["Freeze risky changes, fix reliability first"]
```

## Explained

### 1. SLIs and SLOs
An **SLI** is a measured ratio: good events divided by total events. An **SLO** is the target for that ratio over a window. The Google SRE Workbook advises five or fewer SLIs, a four-week rolling window, and a first target set from recent performance, rounded down so that it is one you can meet. Its page examples are availability of 97% over four weeks, and latency of 90% under 450 ms and 99% under 900 ms.

For LLM serving, the SLIs are TTFT under a threshold, TPOT (or ITL) under a threshold, the success rate, and **goodput** (lesson 07). Keep one set per tenant tier. A single platform-wide number hides the tenant with the longest prompts.

### 2. The error budget and the policy behind it
The **error budget** is 1 minus the SLO, counted in events over the window. A 99% success SLO over four weeks allows 1% of requests to fail. The **error budget policy** is the written part: who wrote it, who approves it, the review dates, what happens when the budget runs out, and who to escalate to. The workbook gives the SLO owner the power to trade feature speed against reliability. The policy is what makes that trade a rule instead of a meeting.

### 3. Who owns what
This split is a design recommendation, built from the SRE Workbook's roles and the Anthropic postmortem. It is not a published org chart.

| Decision | Owner | Why |
|---|---|---|
| Engine image, gateway, routing, autoscaling, dashboards | Platform team | one place to change, shared by all tenants |
| Golden-set eval gate and rollout steps (lesson 35) | Platform team, product sign-off | the same gate for every tenant |
| Prompts and model choice inside the catalogue | Product team | they know the answers they need |
| Tenant budgets and their SLO targets | Product team, approved by the platform | the budget is a product choice; capacity is the platform's |
| Error budget policy and on-call per tier | Written jointly, signed by both | the policy decides who stops a release |

Self-service comes through the gateway: teams and virtual keys with budgets (lessons 29 and 36). Quotas for GPU batch jobs inside a Kubernetes cluster (Kueue) are a Kubernetes topic and are not taught here.

### 4. What changed in the last year
Each row has the date from the release page and the maturity the project states. A version number does not mean that a flag is on by default. Read the release notes for each flag you turn on.

| Item | Date | Maturity (as the project states it) | What it changes |
|---|---|---|---|
| llm-d v1.0.0 | 2026-10-20 | 1.0 release; CNCF Sandbox since March 2026 | flow control, router HA with a hot standby, `DisaggregatedSet`, agent session affinity. Its benchmark numbers are the project's own, on named hardware |
| Dynamo v1.6.0 | 2026-10-22 | versioned release (NVIDIA) | pluggable router flow control; snapshot failover for SGLang and vLLM |
| LeaderWorkerSet v1.0.0 | 2026-10-16 | 1.0 release (a Kubernetes project) | one group of pods as one unit for multi-host inference |
| vLLM v0.32.0 | 2026-10-18 | release; some features are GPU-specific | NVFP4 KV cache on SM8x, SM12x and SM90; GPU n-gram speculator |

| Item | Date | Maturity (as the project states it) | What it changes |
|---|---|---|---|
| vLLM v0.31.0 | 2026-10-05 | release with a **breaking change** | `quantization="fp8"` replaced by `fp8_per_tensor`; draft-model speculation on Model Runner V2 |
| vLLM v0.30.0 | 2026-09-22 | release | HiSparse host-memory KV tier |
| SGLang v0.5.17 to v0.5.22 | 8 Aug to 19 Oct 2026 | releases; v0.5.22 PD speedup is a project claim | PD role switch without restart (0.5.21); up to 27% more throughput on long prompts (0.5.22) |
| Mooncake paper | v5, 2026-10-08 | research paper (arXiv 2407.00079, first posted 2024-06-24) | KV pool and early rejection under overload |

The direction, in one line: KV cache becomes a tiered, shared store; routing reads the cache; disaggregation becomes a default option; and speculative decoding moves into the model's own heads (EAGLE3, MTP). Lesson 40 uses this when it sizes the capstone. Model names in release notes (for example DeepSeek-V4 or GLM-5) can change, so the lessons in this course use small open models.

### 5. Three published write-ups
Each claim below has its source. Each one supports a decision you will make in the capstone (lesson 40).

**DeepSeek, open-infra-index, day 6 (github.com/deepseek-ai/open-infra-index).** Its 24-hour report for 27 to 28 Feb 2025 gives about 73.7k input tokens per second per node in prefill and 14.8k output tokens per second per node in decode. The KV cache hit rate was 56.3% (342B of 608B input tokens). The average was 226.75 nodes of 8 H800 GPUs, with a peak of 278. The cost is $87,072 a day at an assumed $2 per GPU-hour. These are self-reported. *Decision it supports:* size prefill and decode pools separately, and count cache hits as a cost lever.

**Cloudflare, "High-performance LLMs" (blog.cloudflare.com/high-performance-llms/, May 2026).** Prefill and decode disaggregation moved the p90 inter-token latency from about 100 ms with high variance to 20 to 30 ms, about 3x. The `x-session-affinity` header raised the peak input-cache hit ratio from 60% to 80%. The model, Kimi K2.5 (about 560 GB of weights), runs on 8 H100 GPUs. *Decision it supports:* send follow-up turns to the same endpoint, and judge disaggregation by the inter-token latency tail.

**Meta and PyTorch, "Disaggregated Inference at Scale with PyTorch & vLLM" (pytorch.org, 2025-09-12).** Sticky routing with load balancing gave a 40 to 50% prefix-cache hit rate at 90% HBM utilisation. The team used a KV block size of 128 or 256 for transfer, where vLLM's default is 16. Under heavy load, TTFT regressed sharply on TCP, because the network was the bottleneck. *Decision it supports:* put the network in the capacity plan, and measure transfer under load before you choose disaggregation.

Also sourced, not studied in depth: Character.AI's first-party page (opened 2026-10-21) gives about 20,000 queries per second, a serving cost down at least 33x since 2022, and under one cent per hour of conversation. Its technique details come only from third-party summaries, so this course does not use them. Mooncake (arXiv 2407.00079) reports up to 525% throughput in simulated scenarios and 75% more requests in real workloads, both from the abstract.

## Worked example: an error budget and a published cost
Two checks you can do by hand. First, the error budget for the capstone's 10,000 QPS, using made-up failure counts.

In [1]:
qps = 10_000                       # the capstone's sustained request rate (made up for the example)
window_days = 28                   # a four-week rolling window
slo = 0.99                         # 99% of requests must succeed
requests = qps * window_days * 24 * 3600
budget = round(requests * (1 - slo))
failed_so_far = 180_000_000        # made-up count of failed requests in the window so far
print(f"requests in the window: {requests:,}")
print(f"error budget at {slo:.0%}: {budget:,} failed requests")
print(f"budget used so far: {failed_so_far / budget:.0%}, left: {budget - failed_so_far:,}")

requests in the window: 24,192,000,000
error budget at 99%: 241,920,000 failed requests
budget used so far: 74%, left: 61,920,000


Second, the DeepSeek daily cost, rebuilt from its own published inputs. It shows what a published number is made of: arithmetic on an assumed price, not a measured bill.

In [2]:
nodes_avg, gpus_per_node, usd_per_gpu_hour, hours = 226.75, 8, 2.0, 24
cost_per_day = nodes_avg * gpus_per_node * usd_per_gpu_hour * hours
print(f"cost per day at the assumed $2 per GPU-hour: ${cost_per_day:,.0f}")

cost per day at the assumed $2 per GPU-hour: $87,072


The output is 24,192,000,000 requests and an error budget of 241,920,000 failed requests. The made-up failures have used 74% of it, so 61,920,000 are left. The DeepSeek check prints $87,072, which matches its published figure. The agreement shows that the figure is nodes times GPUs times the assumed rate times 24 hours. It does not show what the company paid.

## At real scale
- **The average hides the tenant that hurts.** One platform-wide TTFT looks healthy while one tier with 1,500-token prompts misses its target. Keep SLIs per tenant tier, as section 1 says.
- **Routing changes are quality changes.** In Anthropic's postmortem (2025-09-17), a routing change sent short-context requests to servers built for the 1M-token window. At the peak, 16% of requests in one hour (2025-08-31) were affected, and sticky routing kept follow-up requests on the wrong servers. Gate routing changes like model changes.
- **Numbers to watch:** TTFT and TPOT per tier, success rate, goodput, error budget left, cache hit ratio, and queue depth per replica.
- **Versions move every few weeks.** Pin the version, run the gate from lesson 35 on every bump, and read each release's notes for changed defaults.

## Decision guide

| Situation | Recommended approach | Why |
|---|---|---|
| One team, one model | One written SLO on success and TTFT, no policy | the overhead is not worth more |
| Several tenants on one engine | SLIs and error budget per tenant tier | one tenant's long prompts hide in the average |
| Budget nearly spent | Freeze risky changes (model, flags, quantization); keep reliability fixes flowing | the policy decided this in advance |
| New engine release | Pin the version, run the eval gate and a canary, read the release notes | defaults and formats can change between versions |
| Low cache hits on multi-turn traffic | Session affinity or prefix-aware routing; measure the hit ratio | Cloudflare and Meta both raised hits this way |
| Considering disaggregation | Measure transfer under load on your own network first | Meta found TCP transfer was the bottleneck |
| A vendor benchmark number | Re-measure on your own load (lesson 27) | the number is on their hardware and traffic |

## Common misconceptions
- **"A tighter SLO is always better."** A tighter target leaves less room to change anything, and it costs capacity. The workbook sets the first target from measured performance, rounded down, and the error budget is what buys speed.
- **"A newer version is safer."** vLLM v0.31.0 renamed a quantization option as a breaking change. Newer versions change defaults and formats, and some features exist only on certain GPUs.
- **"A number quoted in many articles is sourced."** Vendor numbers are project-reported. DeepSeek's cost is arithmetic on an assumed rate. A "73%" cost cut attributed to Stripe circulates in explainers without a primary source, so this course does not cite it.
- **"Average latency tells you the platform is healthy."** Use percentiles per tier. Lesson 07 shows why the p95 tail is the number users feel.

## Check yourself
1. A tenant says answers are slow, but the platform's p50 TTFT looks fine. Which measure do you add, and where do you read it?
2. The error budget is 75% used with two weeks of the window left. A team wants to ship a new quantization. What does the policy say, and what evidence do you need first?
3. Session affinity raised Cloudflare's cache hit ratio from 60% to 80%. Name one risk of pinning a session to one endpoint, and the control that covers it.
4. A release note says "new default KV format". What do you check before the next rollout?
5. A platform runs at 2,000 QPS with a 99.5% success SLO over 28 days. How many failed requests does the budget allow?

**Answers**
1. Add a per-tier TTFT SLI and read it from that tier's metrics. The average hides the tail (section 1).
2. The policy applies: no risky change while the budget is low. Bring the lesson 35 gate result on the golden set, a canary plan with a guard, and the approver's sign-off. Reliability fixes continue.
3. One endpoint can overload or fail, and sticky routing can keep requests on a bad pool. Controls: gateway failover and load-aware routing (lesson 30), and a per-endpoint queue limit.
4. Pin the version in a test environment, check the changed default, and run the golden-set gate.
5. 2,000 × 28 × 86,400 = 4,838,400,000 requests, and 0.5% of that is 24,192,000 failed requests.

## Recap
- An SLO is a target on an SLI (good events over total events), kept per tenant tier; the error budget is what the SLO allows to fail, and the policy says what happens when it runs out.
- Ownership is split on purpose: the platform owns engine, routing, gate and dashboards; product teams own prompts, model choice and their tenant targets.
- The last year's changes are dated and rated: versions move fast, flags change defaults, and vendor numbers are self-reported.
- Three write-ups back the design choices: DeepSeek (pool sizing and cache hits), Cloudflare (session affinity and disaggregation), Meta (routing and network).
- **The thing people get wrong:** treating a quoted number as measured on your load. Re-measure it.

## Key terms
- **SLI (service level indicator)** — a measured ratio, good events over total events, such as the share of requests with TTFT under a threshold.
- **Error budget** — 1 minus the SLO, counted in events over the window: the failures the target allows.
- **Error budget policy** — the written rules for an exhausted budget: approvers, review dates, actions and escalation.
- **Session affinity (sticky routing)** — sending every request of one session to the same endpoint so its cached prefix is reused.